# Part 3: Weighted Overlay & Categorization

This notebook calculates suitability scores using AHP weights:
1. **Apply AHP weights** to the reclassified criteria stack
2. **Perform Multi-Criteria Evaluation (MCE) weighted overlay**
3. **Categorize** overlay score into 5 suitability classes (Very Low to Very High)
4. **Analyze final area statistics**
5. **Export spatial GeoTIFF** output

In [ ]:
import os
import sys
import numpy as np
import rasterio

sys.path.append(os.path.abspath('..'))

### 1. Execute Weighted Overlay
We load AHP weights from configuration and run linear weighted combination overlay.

In [ ]:
from src.config import weights
from src.overlay import compute_weighted_overlay

# Compute weighted overlay scores
final_score, valid_all = compute_weighted_overlay(classified_stack, weights)

fv = final_score[~np.isnan(final_score)]
print(f"Overlay complete. Min Score: {fv.min():.4f} | Max Score: {fv.max():.4f} | Mean Score: {fv.mean():.4f}")

### 2. Categorization into 5 classes
Categorize the continuous score (1-9) into Very Low (1), Low (2), Moderate (3), High (4), and Very High (5) classes.

In [ ]:
from src.overlay import categorize_suitability
from src.config import labels, PIXEL_AREA_M2

final_cat = categorize_suitability(final_score)
total_px = int(np.sum(final_cat > 0))

final_class_stats = {}
print("Final Suitability Statistics:")
print(f"{'Class':<12} {'Pixels':>12} {'Area %':>10} {'Area (km²)':>12}")
print("-" * 50)
for c, lbl in labels.items():
    n = int(np.sum(final_cat == c))
    pct = 100 * n / total_px if total_px > 0 else 0
    area_km2 = (n * PIXEL_AREA_M2) / 1e6
    final_class_stats[c] = {
        'label': lbl,
        'pixels': n,
        'pct': round(pct, 4),
        'area_km2': round(area_km2, 4)
    }
    print(f"{lbl:<12} {n:>12,} {pct:>9.2f}% {area_km2:>11.2f}")

### 3. Export to GeoTIFF file

In [ ]:
# Export final suitability classification GeoTIFF
output_dir = '../data/3_validation_map'  # relative local path
os.makedirs(output_dir, exist_ok=True)
output_tif = os.path.join(output_dir, 'Beehive_Suitability_5Class_FINAL.tif')

out_meta = meta.copy()
out_meta.update({'dtype': 'uint8', 'count': 1, 'nodata': 0, 'compress': 'lzw'})

with rasterio.open(output_tif, 'w', **out_meta) as dst:
    dst.write(final_cat, 1)
    dst.update_tags(
        DESCRIPTION='Beehive Suitability 5-Class Map',
        AHP_CR='0.0325',
        RESOLUTION='10m'
    )
print(f"✓ Spatial GeoTIFF successfully saved to: {output_tif}")